# 03.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/03-classification/05-naive-bayes/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings("ignore")

ten = pd.DataFrame({
    "visited":      [1, 1, 1, 1, 0, 1, 0, 0, 0, 1],
    "opened_email": [1, 1, 0, 1, 1, 0, 0, 0, 1, 0],
    "churned":      [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
}, index=range(1, 11))
ten

,visited,opened_email,churned
1,1,1,0
2,1,1,0
3,1,0,0
4,1,1,0
5,0,1,0
6,1,0,0
7,0,0,1
8,0,0,1
9,0,1,1
10,1,0,1


## Exercise 1

Likelihoods needed: $P(V = 1 \mid \text{churn}) = 1/4$ (row 10), $P(E = 0 \mid \text{churn}) = 3/4$; $P(V = 1 \mid \text{stay}) = 5/6$, $P(E = 0 \mid \text{stay}) = 2/6$.

score(churn) = 0.4 × 0.25 × 0.75 = **0.075**; score(stay) = 0.6 × 0.833 × 0.333 = **0.1667**.

P(churn) = 0.075 / (0.075 + 0.1667) = **0.31**. Predict stay: the visit outweighs the unopened email.

In [2]:
from sklearn.naive_bayes import CategoricalNB
X10, y10 = ten[["visited", "opened_email"]], ten["churned"]
sc, ss = 0.4 * 0.25 * 0.75, 0.6 * (5/6) * (2/6)
print("by hand:", round(sc / (sc + ss), 3))
print("sklearn:", CategoricalNB(alpha=0).fit(X10, y10).predict_proba([[1, 0]]).round(3))

by hand: 0.31
sklearn: [[0.69 0.31]]


## Exercise 2

Smoothed likelihoods: churn: $P(V=1) = (1+1)/(4+2) = 0.333$, $P(E=0) = (3+1)/6 = 0.667$; stay: $P(V=1) = (5+1)/(6+2) = 0.75$, $P(E=0) = (2+1)/8 = 0.375$. (scikit-learn also smooths the prior slightly; here the priors are left as 0.4 / 0.6.)

score(churn) = 0.4 × 0.333 × 0.667 = 0.0889; score(stay) = 0.6 × 0.75 × 0.375 = 0.1688. P(churn) = **0.345**, up from 0.31: smoothing pulls every fraction toward 1/2, so extreme counts from tiny samples matter less.

In [3]:
sc, ss = 0.4 * (2/6) * (4/6), 0.6 * (6/8) * (3/8)
print("by hand:", round(sc / (sc + ss), 3))
print("sklearn:", CategoricalNB(alpha=1).fit(X10, y10).predict_proba([[1, 0]]).round(3))

by hand: 0.345
sklearn: [[0.655 0.345]]


## Exercise 3

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
emails = ["item broken want refund", "refund now terrible service", "parcel damaged very disappointed",
          "where is my order thanks", "do you ship to ireland thanks"]
is_complaint = [1, 1, 1, 0, 0]
vec = CountVectorizer(); Xe = vec.fit_transform(emails)
m = MultinomialNB(alpha=1).fit(Xe, is_complaint)
table = pd.DataFrame(np.exp(m.feature_log_prob_).T, index=vec.get_feature_names_out(),
                     columns=["P(word|not)", "P(word|complaint)"]).round(3)
print(table.sort_values("P(word|complaint)", ascending=False))
new = ["my order arrived damaged"]
print("\nP(not, complaint) =", m.predict_proba(vec.transform(new)).round(3))

              P(word|not)  P(word|complaint)
refund              0.031              0.091
broken              0.031              0.061
now                 0.031              0.061
want                0.031              0.061
very                0.031              0.061
terrible            0.031              0.061
service             0.031              0.061
damaged             0.031              0.061
parcel              0.031              0.061
item                0.031              0.061
disappointed        0.031              0.061
order               0.062              0.030
my                  0.062              0.030
is                  0.062              0.030
ship                0.062              0.030
ireland             0.062              0.030
thanks              0.094              0.030
to                  0.062              0.030
do                  0.062              0.030
where               0.062              0.030
you                 0.062              0.030

P(not, co

"damaged" appears only in complaints, so its complaint likelihood is several times its non-complaint one; "order" appears only in a non-complaint and pulls the other way but less strongly; "my", "arrived" are weak or unseen. Net: complaint.

## Exercise 4

In [5]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd, numpy as np
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
from sklearn.model_selection import train_test_split

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")          # fast cache written by lesson 01.3
raw = pd.read_csv(csv, parse_dates=["InvoiceDate"]) if csv.exists() else pd.read_excel(xlsx, sheet_name="Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, accuracy_score, recall_score
Xl_train, Xl_test = X_train.copy(), X_test.copy()
for c in ["spend", "orders", "products"]:
    Xl_train[c], Xl_test[c] = np.log1p(Xl_train[c]), np.log1p(Xl_test[c])
for name, (a, b) in {"raw": (X_train, X_test), "log1p": (Xl_train, Xl_test)}.items():
    p = GaussianNB().fit(a, y_train).predict(b)
    print(f"{name:<6} accuracy {accuracy_score(y_test, p):.3f}  macro F1 {f1_score(y_test, p, average='macro'):.3f}  "
          f"churn F1 {f1_score(y_test, p):.3f}  churn recall {recall_score(y_test, p):.3f}  stay recall {recall_score(y_test, p, pos_label=0):.3f}")

raw    accuracy 0.613  macro F1 0.599  churn F1 0.675  churn recall 0.920  stay recall 0.376
log1p  accuracy 0.673  macro F1 0.673  churn F1 0.665  churn recall 0.743  stay recall 0.620


The churn F1 barely moves, but accuracy and macro F1 jump by six or seven points: the model stops flagging almost everyone (stay recall rises from under 0.4 to over 0.6) and becomes a balanced classifier.

Gaussian NB models each feature as a bell curve per class. Spend and order counts are heavily right-skewed (a few wholesale customers, many small ones), so the fitted curves are wide and badly centred. `log1p` makes those distributions much closer to bell-shaped, which is the only thing the model knows how to represent.